# Day 061 — Exercise 1: setup_logger

Python's `logging` module is the standard way to emit diagnostic output from application code. Unlike `print()`, logging has:

- **Levels** — DEBUG < INFO < WARNING < ERROR < CRITICAL; messages below the logger's level are silently dropped
- **Handlers** — where output goes (stdout, file, network)
- **Formatters** — how output looks
- **Named loggers** — `logging.getLogger(name)` returns a singleton; calling it again with the same name returns the same object

In [ ]:
# --- helper for capturing log records in tests ---
import logging

class _ListHandler(logging.Handler):
    """Stores LogRecord objects in a list for inspection."""
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record):
        self.records.append(record)


In [ ]:
import sys


## Task

Implement `setup_logger(name, level='INFO') -> logging.Logger`:

1. `logging.getLogger(name)` — get (or create) the named logger
2. `logger.setLevel(getattr(logging, level.upper()))` — set numeric level
3. If `not logger.handlers`: add a `StreamHandler(sys.stdout)` with a Formatter
4. `logger.propagate = False` — prevent double-output via the root logger
5. Return the logger

The `if not logger.handlers` guard prevents adding a second handler if the function is called more than once with the same name.

## Your Implementation

In [ ]:
def setup_logger(name: str, level: str = "INFO") -> logging.Logger:
    """Return a configured logger.

    - getLogger(name) — named logger (singleton per name)
    - setLevel to the numeric value of level (e.g. 'WARNING' -> logging.WARNING)
    - Add a StreamHandler(sys.stdout) with a basic Formatter IF no handlers exist
    - Set logger.propagate = False so root logger doesn't duplicate output
    - Return the logger
    """
    # TODO: configure and return the logger
    raise NotImplementedError


In [ ]:
def setup_logger(name: str, level: str = "INFO") -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level.upper()))
    if not logger.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter(
            "%(asctime)s | %(name)s | %(levelname)s | %(message)s"
        ))
        logger.addHandler(h)
    logger.propagate = False
    return logger


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # create logger at WARNING level
    logger = setup_logger("day061_ex1", "WARNING")

    # wire in our capturing handler
    h = _ListHandler()
    logger.handlers.clear()
    logger.addHandler(h)
    logger.propagate = False

    # check level
    assert logger.level == logging.WARNING, (
        f"Expected WARNING ({logging.WARNING}), got {logger.level}")
    score += 1; print("\u2705 logger.level is WARNING")

    # DEBUG below WARNING — should not log
    logger.debug("ignored")
    assert len(h.records) == 0, "DEBUG should be filtered by WARNING level"
    score += 1; print("\u2705 DEBUG below WARNING is filtered")

    # INFO below WARNING — should not log
    logger.info("also ignored")
    assert len(h.records) == 0, "INFO should be filtered by WARNING level"
    score += 1; print("\u2705 INFO below WARNING is filtered")

    # WARNING logs
    logger.warning("first warning")
    assert len(h.records) == 1
    assert h.records[0].levelname == "WARNING"
    assert "first warning" in h.records[0].getMessage()
    score += 1; print("\u2705 WARNING level emits a log record")

    # ERROR logs too
    logger.error("something broke")
    assert len(h.records) == 2
    assert h.records[1].levelname == "ERROR"
    score += 1; print("\u2705 ERROR level emits a log record")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def setup_logger(name: str, level: str = "INFO") -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level.upper()))
    if not logger.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter(
            "%(asctime)s | %(name)s | %(levelname)s | %(message)s"
        ))
        logger.addHandler(h)
    logger.propagate = False
    return logger
```

**Why `propagate = False`?** By default, log records bubble up to the root logger. If the root logger also has a handler (e.g. from `logging.basicConfig`), every message appears twice. Setting `propagate = False` stops the bubble-up.

</details>